# Hansen Ch.20 Series Regression

理论逐步证明见 `Hansen_Ch20_Exercises_Solutions.md`（**20.1–20.18**）。

本 notebook：多项式 / 样条 / 部分线性 / 2SLS-NPIV 实证，**代码含中文注释**。

In [ ]:

# Ch.20 series regression utilities
# 详尽注释：基函数构造、OLS、AIC、LOOCV、cluster SE

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import pinv, inv
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")


def ols(y, X, cluster=None):
    """OLS with optional cluster-robust sandwich variance.
    y: (n,), X: (n,k) already including intercept if needed.
    Returns beta, se, V, residuals.
    """
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    b = pinv(X.T @ X) @ (X.T @ y)
    e = y - X @ b
    n, k = X.shape
    if cluster is None:
        # classical homoskedastic V = s^2 (X'X)^{-1}
        s2 = (e @ e) / (n - k)
        V = s2 * pinv(X.T @ X)
    else:
        # Arellano-type cluster meat
        meat = np.zeros((k, k))
        for g in pd.unique(cluster):
            m = cluster == g
            score = X[m].T @ e[m]
            meat += np.outer(score, score)
        nG = len(pd.unique(cluster))
        bread = pinv(X.T @ X)
        V = (nG / (nG - 1)) * ((n - 1) / (n - k)) * bread @ meat @ bread
    se = np.sqrt(np.maximum(np.diag(V), 0.0))
    return b, se, V, e


def aic(y, X):
    """Gaussian AIC = n log(SSR/n) + 2k  (up to additive constant)."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    n, k = X.shape
    e = y - X @ (pinv(X.T @ X) @ (X.T @ y))
    s2 = max((e @ e) / n, 1e-300)
    return n * np.log(s2) + 2 * k


def loocv(y, X):
    """Leave-one-out CV via leverage formula: e_i / (1-h_ii)."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    XtXinv = pinv(X.T @ X)
    # h_ii = x_i' (X'X)^{-1} x_i
    Hdiag = np.sum((X @ XtXinv) * X, axis=1)
    b = XtXinv @ (X.T @ y)
    e = y - X @ b
    return np.mean((e / (1 - Hdiag)) ** 2)


def poly_basis(x, p):
    """Polynomial basis 1, x, ..., x^p. x should already be scaled to [0,1]."""
    x = np.asarray(x, float)
    return np.column_stack([x ** j for j in range(p + 1)])


def linear_spline(x, knots):
    """Linear spline: 1, x, (x-tau)_+ for each knot."""
    x = np.asarray(x, float)
    cols = [np.ones_like(x), x]
    for t in knots:
        cols.append(np.maximum(x - t, 0.0))
    return np.column_stack(cols)


def quad_spline(x, knots):
    """Quadratic spline: 1, x, x^2, (x-tau)_+^2 for each knot."""
    x = np.asarray(x, float)
    cols = [np.ones_like(x), x, x ** 2]
    for t in knots:
        cols.append(np.maximum(x - t, 0.0) ** 2)
    return np.column_stack(cols)


def m_and_se_at_grid(b, V, basis_rows):
    """Given beta, Var(beta), and list/array of a(x)' rows, return m(x) and se."""
    m = basis_rows @ b
    # se^2 = a' V a for each row
    se = np.sqrt(np.maximum(np.sum(basis_rows * (basis_rows @ V), axis=1), 0.0))
    return m, se


## 20.1 数值核对：X=3 的边际效应 = 4

In [ ]:
# m'(x) = 2 + 5*1{x>=1} - 3*1{x>=2}
for x in [0.5, 1.5, 3.0]:
    me = 2 + 5 * (x >= 1) - 3 * (x >= 2)
    print(f"x={x}, ME={me}")


## 20.9–20.14 CPS 工资剖面

In [ ]:
cps = pd.read_excel(ROOT / "cps09mar/cps09mar.xlsx")
# experience 与 log 小时工资
cps["exp"] = cps["age"] - cps["education"] - 6
cps["lw"] = np.log(cps["earnings"] / (cps["hours"] * cps["week"]))
cps = cps.replace([np.inf, -np.inf], np.nan).dropna(subset=["lw", "exp", "education"])
cps = cps[(cps["exp"] >= 0) & (cps["exp"] <= 70)]
y = cps["lw"].values
x = cps["exp"].values.astype(float)
xe = cps["education"].values.astype(float)

# 重标到 [0,1]，减轻高次幂共线（Hansen §20.4）
def unit(z):
    zmin, zmax = z.min(), z.max()
    return (z - zmin) / (zmax - zmin + 1e-15), zmin, zmax

xr, xmin, xmax = unit(x)
print("n =", len(y), "exp range", xmin, xmax)

print("\nPoly order selection for experience:")
for p in range(1, 9):
    X = poly_basis(xr, p)
    print(f"  p={p}: AIC={aic(y,X):.1f}, LOOCV={loocv(y,X):.6f}")

# 6th order poly + pointwise band at integer experience
X = poly_basis(xr, 6)
b, se, V, e = ols(y, X)
grid = np.arange(0, 61)
gr = (grid - xmin) / (xmax - xmin + 1e-15)
A = poly_basis(gr, 6)
m, sm = m_and_se_at_grid(b, V, A)
print("\n6th poly log wage at exp 10,20,40:")
for g, mi, si in zip(grid, m, sm):
    if g in (10, 20, 40, 50, 60):
        print(f"  exp={g}: m={mi:.3f}, se={si:.3f}, 95% CI=[{mi-1.96*si:.3f},{mi+1.96*si:.3f}]")

print("\nQuadratic spline (experience) AIC/CV:")
for name, knots in [("none", []), ("@20", [20]), ("@20,40", [20, 40]), ("@10,20,30,40", [10, 20, 30, 40])]:
    Xs = quad_spline(x, knots)
    print(f"  {name}: AIC={aic(y,Xs):.1f}, CV={loocv(y,Xs):.6f}")

print("\nQuadratic spline (education) AIC/CV:")
for name, knots in [("none", []), ("@10", [10]), ("@5,10,15", [5, 10, 15]), ("@4,8,12,16", [4, 8, 12, 16])]:
    Xs = quad_spline(xe, knots)
    print(f"  {name}: AIC={aic(y,Xs):.1f}, CV={loocv(y,Xs):.6f}")


## 20.15 RR2010 部分线性债务

In [ ]:
rr = pd.read_excel(ROOT / "RR2010/RR2010.xlsx").sort_values("year")
rr["Y"] = pd.to_numeric(rr["gdp"], errors="coerce")
rr["D"] = pd.to_numeric(rr["debt"], errors="coerce")
rr["Ylag"] = rr["Y"].shift(1)
rr["Dlag"] = rr["D"].shift(1)
d = rr.dropna(subset=["Y", "Ylag", "Dlag"])
Y, Ylag, Dlag = d["Y"].values, d["Ylag"].values, d["Dlag"].values

for name, knots in [("linear m", None), ("knot 60", [60]), ("knots 40,80", [40, 80])]:
    if knots is None:
        X = np.column_stack([np.ones(len(Y)), Ylag, Dlag])
    else:
        # Y_t = alpha Y_{t-1} + linear_spline(D_{t-1}) + e
        X = np.column_stack([Ylag, linear_spline(Dlag, knots)])
    print(f"{name}: AIC={aic(Y,X):.2f}, CV={loocv(Y,X):.3f}")

X = np.column_stack([Ylag, linear_spline(Dlag, [60])])
b, se, V, e = ols(Y, X)
print("one-knot coeffs [alpha, b0, b1, b2]:", b)
print("se:", se)
print("slope D<60:", b[2], "; slope D>=60:", b[2] + b[3])


## 20.16 DDK 百分位二次样条

In [ ]:
ddk = pd.read_excel(ROOT / "DDK2011/DDK2011.xlsx")
x = pd.to_numeric(ddk["percentile"], errors="coerce")
y = pd.to_numeric(ddk["r2_totalscore"], errors="coerce")
cl = ddk["schoolid"].values
m = np.isfinite(x) & np.isfinite(y)
x, y, cl = x[m].values, y[m].values, cl[m]
for name, knots in [
    ("none", []),
    ("@50", [50]),
    ("@33,66", [33, 66]),
    ("@25,50,75", [25, 50, 75]),
    ("@20,40,60,80", [20, 40, 60, 80]),
]:
    X = quad_spline(x, knots)
    print(f"{name}: AIC={aic(y,X):.1f}, CV={loocv(y,X):.3f}")
# preferred: one knot at 50, cluster SE
X = quad_spline(x, [50])
b, se, V, e = ols(y, X, cluster=cl)
grid = np.linspace(1, 99, 20)
# build basis at grid
A = quad_spline(grid, [50])
mhat, se_m = m_and_se_at_grid(b, V, A)
print("selected model m(percentile) with cluster SE (sample points):")
for g, mi, si in zip(grid[::4], mhat[::4], se_m[::4]):
    print(f"  p={g:.0f}: {mi:.2f} ({si:.2f})")


## 20.17 CHJ2004 transfers ~ poly(income)

In [ ]:
chj = pd.read_stata(ROOT / "CHJ2004/CHJ2004.dta")
y = pd.to_numeric(chj["transfers"], errors="coerce").values
inc = pd.to_numeric(chj["income"], errors="coerce").values
m = np.isfinite(y) & np.isfinite(inc) & (inc >= 0)
y, inc = y[m], inc[m]
# scale income to [0,1]
ir = (inc - inc.min()) / (inc.max() - inc.min() + 1e-15)
print("CHJ n=", len(y))
for p in range(1, 9):
    X = poly_basis(ir, p)
    print(f"p={p}: AIC={aic(y,X):.1f}, CV={loocv(y,X):.1f}")
# with demographic controls
ctrl = ["primary", "somesecondary", "secondary", "someuniversity", "university",
        "age", "female", "married", "child1", "child7", "child15", "size",
        "bothwork", "notemployed", "marriedf"]
df = chj.loc[m].copy()
# rebuild boolean mask index
# simpler: use full frame cleaned
df = chj.copy()
df["transfers"] = pd.to_numeric(df["transfers"], errors="coerce")
df["income"] = pd.to_numeric(df["income"], errors="coerce")
df = df.dropna(subset=["transfers", "income"])
df = df[df["income"] >= 0]
ir = (df["income"] - df["income"].min()) / (df["income"].max() - df["income"].min() + 1e-15)
Xpoly = poly_basis(ir.values, 6)
Xc = df[ctrl].apply(pd.to_numeric, errors="coerce").fillna(0).values
X = np.column_stack([Xpoly, Xc])
b, se, V, e = ols(df["transfers"].values, X)
print("poly6 + controls: first 7 poly coefs", b[:7])


## 20.18 AL1999 math scores NPIV-style 2SLS

In [ ]:
al = pd.read_excel(ROOT / "AL1999/AL1999.xlsx")
enr = pd.to_numeric(al["enrollment"], errors="coerce")
# Maimonides predicted class size p = enrollment / nclass
nclass = 1 + np.floor((enr - 1) / 40.0)
al["p"] = enr / nclass
for c in ["classize", "disadvantaged", "avgmath", "enrollment"]:
    al[c] = pd.to_numeric(al[c], errors="coerce")
al["grade4"] = (pd.to_numeric(al["grade"], errors="coerce") == 4).astype(float)
df = al.dropna(subset=["avgmath", "classize", "disadvantaged", "enrollment", "p", "schlcode"]).copy()

c = df["classize"].values / 40.0
d = df["disadvantaged"].values / 14.0
Y = df["avgmath"].values
# endogenous: class size poly + interaction
X_en = np.column_stack([c, c**2, c**3, c * d])
# exogenous controls
X_ex = np.column_stack([np.ones(len(df)), d, d**2, d**3, df["enrollment"].values, df["grade4"].values])
X = np.column_stack([X_en, X_ex])
# instruments for endogenous parts
pv = df["p"].values / 40.0
Z_iv = np.column_stack([pv, pv**2, pv**3, pv * d])
Z = np.column_stack([Z_iv, X_ex])

# 2SLS: project X onto Z then OLS
Pz_X = Z @ (pinv(Z.T @ Z) @ (Z.T @ X))
b = pinv(Pz_X.T @ Pz_X) @ (Pz_X.T @ Y)
e = Y - X @ b
# cluster sandwich using projected X
cl = df["schlcode"].values
meat = np.zeros((len(b), len(b)))
for g in pd.unique(cl):
    msk = cl == g
    score = Pz_X[msk].T @ e[msk]
    meat += np.outer(score, score)
n, k = X.shape
nG = len(pd.unique(cl))
bread = pinv(Pz_X.T @ Pz_X)
V = (nG / (nG - 1)) * ((n - 1) / (n - k)) * bread @ meat @ bread
se = np.sqrt(np.maximum(np.diag(V), 0))
names = ["c", "c2", "c3", "c*d", "int", "d", "d2", "d3", "enr", "g4"]
print("n =", n)
for nm, bi, si in zip(names, b, se):
    print(f"{nm:6s} {bi:9.3f} ({si:.3f})")

# effect of classize 20 -> 40 at mean disadvantaged
dbar = d.mean()
a = np.zeros(len(b))
a[0] = 0.5
a[1] = 0.75
a[2] = 0.875
a[3] = 0.5 * dbar
th = a @ b
print(f"impact 20->40: {th:.3f} (se {np.sqrt(a@V@a):.3f})")
R = np.eye(4, len(b))
W = float((R @ b) @ inv(R @ V @ R.T) @ (R @ b))
print(f"Wald (4 class terms)={W:.2f}, p={1-stats.chi2.cdf(W,4):.3f}")
